<a href="https://colab.research.google.com/github/wtwwgegric/SDSC4070-Homework2/blob/main/SDSC4070_ass2_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetune Qwen2.5 with LLaMA Factory

Name: Yin Taihua
SID: 58532351

## 1.1 Install Dependencies

In [ ]:
%cd /content/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]
!pip install -r requirements/metrics.txt

/content
Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 636, done.
remote: Counting objects: 100% (636/636), done.
remote: Compressing objects: 100% (479/479), done.
remote: Total 636 (delta 152), reused 404 (delta 97), pack-reused 0 (from 0)
Receiving objects: 100% (636/636), 5.28 MiB | 19.30 MiB/s, done.
Resolving deltas: 100% (152/152), done.
/content/LLaMA-Factory
assets/       docker/    LICENSE      pyproject.toml  requirements/  tests/
CITATION.cff  docs/      Makefile     README.md       scripts/       tests_v1/
data/         examples/  MANIFEST.in  README_zh.md    src/
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

### 1.2 Check GPU environment

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Using device: {device}")
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")

Using device: cuda


## 2.1 Clean Dataset

In [ ]:
from datasets import load_dataset
import json
import os
import re
from collections import Counter
import random

%cd /content/LLaMA-Factory/

# 1. clean useless label, tag
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Clean HTML label
    text = re.sub(r'<[^>]+>', '', text)
    # Only retain valid characters
    text = re.sub(r'[^A-Za-z0-9\u4e00-\u9fa5（）().,，!！？；;:：、\n -]+', ' ', text)
    # Clean up extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 2. Download IMDB dataset（1000 records in random）
full_train = load_dataset("imdb", split="train")

positive_indices = [i for i, item in enumerate(full_train) if item["label"] == 1]
negative_indices = [i for i, item in enumerate(full_train) if item["label"] == 0]

random.seed(42)
selected_indices = random.sample(positive_indices, 500) + random.sample(negative_indices, 500)
balanced_train = full_train.select(selected_indices)

print(f"------* Label Distribution: {Counter(balanced_train['label'])} *------")

imdb = balanced_train

# 3. Transform into Alpaca format
alpaca_data = []
invalid_count = 0
for i, item in enumerate(imdb):
    # clean labels
    cleaned_text = clean_text(item["text"])
    sentiment = "Positive" if item["label"] == 1 else "Negative"

    # Skip samples that are empty after cleaning
    if len(cleaned_text) < 10:
        invalid_count += 1
        continue
    # transform into alpaca format
    alpaca_data.append({
        "instruction": "Classify the sentiment of this movie review as Positive or Negative.",
        "input": cleaned_text,
        "output": sentiment,
        "system": ""
    })

print(f"Data transformed: {len(alpaca_data)}")
print(f"Data skipped: {invalid_count} ")

# 4. Save as JSON file
os.makedirs("data/my_imdb", exist_ok=True)
with open("data/my_imdb/train.json", "w", encoding="utf-8") as f:
    json.dump(alpaca_data, f, ensure_ascii=False, indent=2)

# content/LLaMA-Factory/data/my_imdb/train.json

# 5. Show 3 random sample to double check
print("\nSample:")
random.seed(114514)
for i in range(0,3):
    print(f"\n ### Sample {random.randint(1, 1000)} ###")
    print(f"Instruction: {alpaca_data[random.randint(1, 1000)]['instruction']}...")
    print(f"Input: {alpaca_data[random.randint(1, 1000)]['input'][:100]}...")
    print(f"Output: {alpaca_data[random.randint(1, 1000)]['output']}")

/content/LLaMA-Factory
------* Label Distribution: Counter({1: 500, 0: 500}) *------
Data transformed: 1000
Data skipped: 0 

Sample:

 ### Sample 239 ###
Instruction: Classify the sentiment of this movie review as Positive or Negative....
Input: I guess if you are into the sci-fi and horror stuff it might be interesting. The acting was okay but...
Output: Positive

 ### Sample 164 ###
Instruction: Classify the sentiment of this movie review as Positive or Negative....
Input: Live Feed is set in some unnamed Chinese Japanese Asian district somewhere as five American friends,...
Output: Negative

 ### Sample 296 ###
Instruction: Classify the sentiment of this movie review as Positive or Negative....
Input: Although a tear jerker it is definitely a feel good movie. All the actors were excellent and Will Sm...
Output: Negative


## 2.2 Update IMDB dataset

In [ ]:
import json

# read "dataset_info"
dataset_info_path = "data/dataset_info.json"
with open(dataset_info_path, "r", encoding="utf-8") as f:
    dataset_info = json.load(f)

# Add IMDB（Alpaca style）
dataset_info["imdb_alpaca"] = {
    "file_name": "my_imdb/train.json"
}

# write into "dataset_info"
with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

## 3.1 SFT config

### Fine-tune model via LLaMA WebUI

In [ ]:
%cd /content/LLaMA-Factory/
!GRADIO_SHARE=1 llamafactory-cli webui

/content/LLaMA-Factory
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
Visit http://ip:port for Web UI, e.g., http://127.0.0.1:7860
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://c4503b718a1f97d7e0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
[WARNING|2026-03-25 15:27:51] llamafactory.webui.common:149 >> Found complex path, some feat

### Fine-tune model via Command Line

It takes ~30min for training.

In [ ]:
import yaml
import os

os.makedirs("/content/result", exist_ok=True)

# --- Full Fine-tuning Config. ---
full_config = {
    "stage": "sft",
    "do_train": True,
    "model_name_or_path": "Qwen/Qwen2.5-0.5B-Instruct",
    "dataset": "imdb_alpaca",
    "template": "qwen",
    "finetuning_type": "full",
    "output_dir": "/content/result/full",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "gradient_checkpointing": True,
    "learning_rate": 1e-5,
    "num_train_epochs": 3.0,
    "max_samples": 1000,
    "fp16": True,
    "report_to": "none",
    "overwrite_output_dir": True,
    "logging_steps": 5,
    "save_steps": 100,
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "cosine",
    "max_grad_norm": 1.0,
    "val_size": 0.1,              # 10% data as validation
    "eval_steps": 100,            # evaluate every 100 steps
    "eval_strategy": "steps"
}

with open("train_full.yaml", "w", encoding="utf-8") as f:
    yaml.dump(full_config, f, allow_unicode=True, default_flow_style=False)



In [ ]:
# Full Fine-tuning
!llamafactory-cli train /content/LLaMA-Factory/train_full.yaml

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-03-26 06:21:02] llamafactory.hparams.parser:505 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
config.json: 100% 659/659 [00:00<00:00, 3.22MB/s]
[INFO|configuration_utils.py:667] 2026-03-26 06:21:02,944 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae

## 3.2 LoRA

In [ ]:
# --- PEFT ~ LoRA finetuning config ---
lora_config = full_config.copy()
lora_config.update({
    "finetuning_type": "lora",
    "lora_target": "all",
    "learning_rate": 1e-4,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "output_dir": "/content/result/lora",
})

with open("train_lora.yaml", "w", encoding="utf-8") as f:
    yaml.dump(lora_config, f, allow_unicode=True, default_flow_style=False)



In [ ]:

# LoRA fine-tuning
!llamafactory-cli train /content/LLaMA-Factory/train_lora.yaml

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-03-26 06:31:50] llamafactory.hparams.parser:505 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:667] 2026-03-26 06:31:51,196 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c2c2f167def9a775/config.json
[INFO|configuration_utils.py:739] 2026-03-26 06:31:51,199 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_a

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

os.makedirs('/content/drive/MyDrive/llama_exps', exist_ok=True)

# copy result into Drive
shutil.copytree('/content/result/full', '/content/drive/MyDrive/llama_exps/output_full', dirs_exist_ok=True)
shutil.copytree('/content/result/lora', '/content/drive/MyDrive/llama_exps/output_lora', dirs_exist_ok=True)


'/content/drive/MyDrive/llama_exps/output_lora'

## Infer the fine-tuned model

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00


In [ ]:
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc
from google.colab import drive
from datasets import load_dataset
from evaluate import load
from sklearn.metrics import classification_report
from collections import Counter
import random

drive.mount('/content/drive')
%cd /content/LLaMA-Factory/

full_test = load_dataset("imdb", split="test").shuffle(seed=42)
positive_indices = [i for i, item in enumerate(full_test) if item["label"] == 1]
negative_indices = [i for i, item in enumerate(full_test) if item["label"] == 0]
selected_indices = random.sample(positive_indices, 100) + random.sample(negative_indices, 100)
test_data = full_test.select(selected_indices)

print(f"Test Set label distibution: {Counter(test_data['label'])}")

def evaluate(adapter_path, finetuning_type):
    if finetuning_type == "full":
        args = dict(model_name_or_path=adapter_path,
                    adapter_name_or_path=None,
                    template="qwen",
                    finetuning_type="full",
                    infer_dtype="float16")
    else:
        args = dict(model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",
                    adapter_name_or_path=adapter_path, template="qwen",
                    finetuning_type="lora",
                    infer_dtype="float16")

    chat_model = ChatModel(args)
    predictions, references = [], []

    for i, item in enumerate(test_data):

        query = "Classify the sentiment of this movie review as Positive or Negative." + item['text'][:500]
        messages = [{"role": "user", "content": query}]

        response = chat_model.chat(messages=messages)
        response_text = response[0].response_text.strip().lower()

        if response_text == "positive":
            pred = 1
        elif response_text == "negative":
            pred = 0
        else:
            continue

        predictions.append(pred)
        references.append(item["label"])

        if i < 10:
            status = "✅" if pred == item["label"] else "❌"
            print(f"{status} [{i}] True={item['label']}, Pred={pred}, Output='{response_text}'")

    accuracy = load("accuracy").compute(predictions=predictions, references=references)["accuracy"]
    report = classification_report(references, predictions, target_names=["Negative", "Positive"], output_dict=True)

    print(f"\n Accuracy: {accuracy:.4f}")
    print(f"F1-Score (Positive): {report['Positive']['f1-score']:.4f}")

    return {"accuracy": accuracy, "f1": report['Positive']['f1-score']}

print("="*60)
print("Full Fine-tuning")
print("="*60)
full_results = evaluate("/content/drive/MyDrive/llama_exps/output_full", "full")

print("\n" + "="*60)
print("LoRA")
print("="*60)
lora_results = evaluate("/content/drive/MyDrive/llama_exps/output_lora", "lora")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/LLaMA-Factory


[INFO|configuration_utils.py:665] 2026-03-26 07:00:36,272 >> loading configuration file /content/drive/MyDrive/llama_exps/output_full/config.json
[INFO|configuration_utils.py:739] 2026-03-26 07:00:36,274 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": null,
  "dtype": "float32",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_

Test Set label distibution: Counter({1: 100, 0: 100})
Full Fine-tuning


[INFO|configuration_utils.py:665] 2026-03-26 07:00:37,887 >> loading configuration file /content/drive/MyDrive/llama_exps/output_full/config.json
[INFO|configuration_utils.py:739] 2026-03-26 07:00:37,889 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": null,
  "dtype": "float32",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_

[INFO|2026-03-26 07:00:39] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


...

[INFO|2026-03-26 07:01:02] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


[INFO|modeling_utils.py:732] 2026-03-26 07:01:02,913 >> loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c2c2f167def9a775/model.safetensors
[INFO|modeling_utils.py:801] 2026-03-26 07:01:02,914 >> Will use dtype=torch.bfloat16 as defined in model's config object
[INFO|configuration_utils.py:1014] 2026-03-26 07:01:02,916 >> Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

[INFO|accelerate.py:214] 2026-03-26 07:01:02,958 >> We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[INFO|configuration_utils.py:967] 2026-03-26 07:01:03,719 >> loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c2c2f167def9a775/generation_config.json
[INFO|configuration_utils.py:1014] 2026-03-26 07:01:03,720 >> Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



[INFO|2026-03-26 07:01:03] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-03-26 07:01:04] llamafactory.model.adapter:144 >> Merged 1 adapter(s).
[INFO|2026-03-26 07:01:04] llamafactory.model.adapter:144 >> Loaded adapter(s): /content/drive/MyDrive/llama_exps/output_lora
[INFO|2026-03-26 07:01:04] llamafactory.model.loader:144 >> all params: 494,032,768
❌ [0] True=1, Pred=0, Output='negative'
✅ [1] True=1, Pred=1, Output='positive'
✅ [2] True=1, Pred=1, Output='positive'
✅ [3] True=1, Pred=1, Output='positive'
✅ [4] True=1, Pred=1, Output='positive'
❌ [5] True=1, Pred=0, Output='negative'
✅ [6] True=1, Pred=1, Output='positive'
❌ [7] True=1, Pred=0, Output='negative'
✅ [8] True=1, Pred=1, Output='positive'
✅ [9] True=1, Pred=1, Output='positive'

 Accuracy: 0.8750
F1-Score (Positive): 0.8744
